# DeiT-S from Scratch on ImageNet-100

> This notebook trains a **DeiT-S / ViT-S/16** image classifier **from scratch** (no pre-trained
> weights) on **ImageNet-100**, a 100-class subset of ILSVRC-2012 (~130k train / 5k val images).
>
> The architecture and the training recipe both follow
> [*Training data-efficient image transformers & distillation through attention*](https://arxiv.org/abs/2012.12877)
> (Touvron et al., 2021): patch 16, embed dim 384, depth 12, 6 heads, ~21.7M parameters.
>
> The model defaults to `timm.create_model('deit_small_patch16_224', pretrained=False)` - the
> exact ImageNet-1k DeiT-S graph with random weights. A hand-written implementation using only
> fundamental PyTorch modules (`nn.Linear`, `nn.Conv2d`, `nn.LayerNorm`, ...) sits behind
> `config['model_source'] = 'scratch'`; the two are bit-identical (same 152 state-dict tensors,
> identical outputs for identical weights), so you can switch either way.
>
> The data pipeline, Mixup, loss and LR schedule are taken from timm with DeiT's own arguments,
> rather than re-implemented, so the recipe matches `deit/main.py` rather than approximating it.

## What this notebook does

| Stage | Contents |
|---|---|
| Set-up | one `PLATFORM` flag for Kaggle / Colab / PSC / local: installs, data, paths |
| Data | ImageNet-100 folder parsing, DeiT augmentation recipe, MixUp / CutMix |
| Model | timm `deit_small_patch16_224` (or the bit-identical from-scratch implementation) |
| Training | DeiT recipe: AdamW, timm cosine schedule, RandAugment, Mixup/CutMix, repeated aug, EMA, AMP |
| Eval | Top-1 / Top-5, per-class accuracy, hflip TTA, prediction export |
| Resume | Every epoch is checkpointed so training survives Kaggle's session limit |

**Vision transformers are data hungry.** On 130k images, expect roughly **72-80% top-1** for
DeiT-S at 224px after 100-300 epochs with the full augmentation recipe. Short runs (<30 epochs)
will look disappointing - that is expected behaviour for ViTs trained from scratch, not a bug.

# **SET-UP**

## Step 0: Choose your platform

This notebook runs unchanged on **Kaggle**, **Colab**, **PSC Bridges-2** or a local machine.
Set the `PLATFORM` flag in the next cell and everything downstream - package installs, dataset
location, checkpoint directory, output directory - is wired up for you.

| `PLATFORM` | Data comes from | Checkpoints & outputs go to |
|---|---|---|
| `'kaggle'` | a dataset you mounted under `/kaggle/input` | `/kaggle/working` |
| `'colab'`  | downloaded to `/content` via the Kaggle API | Google Drive (survives disconnects) |
| `'psc'`    | downloaded to node-local `$LOCAL` | `~/imagenet100_deit` on `/jet/home` |
| `'local'`  | `$IMAGENET100_DIR` or `./imagenet100` | `./work` |
| `'auto'`   | detected from the environment | as above |

Leave it on `'auto'` unless you want to force a specific behaviour.

The split on PSC is deliberate and the one thing worth understanding there: the dataset goes on
**node-local `$LOCAL`** because reading 130k small JPEGs from the shared filesystem is slow
enough to add hours per epoch, while checkpoints go to **`/jet/home`** because `$LOCAL` is wiped
when your job ends or you land on a different node.

In [ ]:
# ============================ THE ONLY FLAG YOU NEED ============================
PLATFORM = 'auto'              # 'auto' | 'kaggle' | 'colab' | 'psc' | 'local'
# ===============================================================================

# ---- Colab options (ignored on other platforms) ----
COLAB_MOUNT_DRIVE  = True                  # keep checkpoints on Drive so they survive disconnects
COLAB_DRIVE_SUBDIR = 'imagenet100_deit'    # folder created inside MyDrive
COLAB_DATA_DIR     = '/content/imagenet100'  # local SSD is far faster than reading from Drive

# ---- PSC / Bridges-2 options (ignored on other platforms) ----
PSC_WORK_SUBDIR = 'imagenet100_deit'       # created under your persistent /jet/home

# ---- dataset to download when the platform has no mounted copy (Colab / PSC / local) ----
KAGGLE_DATASET = 'ambityga/imagenet100'

import os
import sys
import importlib.util
from pathlib import Path


def psc_scratch():
    # Node-local scratch on Bridges-2; $LOCAL is set for you on the compute node.
    return os.environ.get('LOCAL', '/local')


def detect_platform():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.isdir('/kaggle/input'):
        return 'kaggle'
    if importlib.util.find_spec('google.colab') is not None:
        return 'colab'
    if os.path.isdir('/jet/home') or os.environ.get('LOCAL', '').startswith('/local'):
        return 'psc'
    return 'local'


PLATFORM = detect_platform() if PLATFORM == 'auto' else PLATFORM
assert PLATFORM in ('kaggle', 'colab', 'psc', 'local'), f"unknown PLATFORM: {PLATFORM}"
IS_KAGGLE = PLATFORM == 'kaggle'
IS_COLAB  = PLATFORM == 'colab'
IS_PSC    = PLATFORM == 'psc'
IS_LOCAL  = PLATFORM == 'local'


def resolve_work_dir():
    """Where checkpoints, logs and exports are written."""
    if IS_KAGGLE:
        return '/kaggle/working'
    if IS_COLAB:
        if COLAB_MOUNT_DRIVE:
            try:
                from google.colab import drive
                drive.mount('/content/drive')
                d = f"/content/drive/MyDrive/{COLAB_DRIVE_SUBDIR}"
                os.makedirs(d, exist_ok=True)
                return d
            except Exception as e:
                print(f"[warn] Drive mount failed ({e}); falling back to /content/work. "
                      f"Download checkpoints before the runtime disconnects!")
        return '/content/work'
    if IS_PSC:
        # /jet/home survives between jobs and nodes; $LOCAL does not, so checkpoints must
        # not live there even though the dataset should.
        return os.path.expanduser(f"~/{PSC_WORK_SUBDIR}")
    return './work'


ENV = {
    'platform' : PLATFORM,
    'work_dir' : resolve_work_dir(),
    'data_root': None,            # filled in by the 'Get the data' step below
}
os.makedirs(ENV['work_dir'], exist_ok=True)

print(f"Platform : {PLATFORM}")
print(f"Work dir : {ENV['work_dir']}")
if IS_PSC:
    scratch = psc_scratch()
    print(f"Scratch  : {scratch} (node-local, cleared when you change nodes)")
    if not os.path.isdir(scratch) or not os.access(scratch, os.W_OK):
        print("[warn] $LOCAL is missing or not writable. You are probably on a login node - "
              "request a GPU node with `interact` first (see the PSC setup cell below).")

## Step 1: Verify the GPU

- **Kaggle**: *Notebook → Settings → Accelerator → GPU T4 x2* (preferred; the training loop
  uses both cards automatically) or P100.
- **Colab**: *Runtime → Change runtime type → T4 GPU*. A single T4 is roughly half the
  throughput of Kaggle's T4 x2, so expect ~2x the epoch time.
- **PSC**: you must already be on a GPU compute node (see the PSC setup below). A single
  `v100-32` is a good deal faster than a T4 for this model.

In [ ]:
!nvidia-smi

## PSC / Bridges-2 setup (skip on Kaggle and Colab)

Everything above and below runs the same way on PSC, but you have to get onto a GPU node and
start a Jupyter server first. From a login shell:

```bash
# 1. a GPU node (8 h is the usual limit; match config['max_runtime_hours'] to it)
interact -p GPU-shared --gres=gpu:v100-32:1 -t 8:00:00 -A <your_allocation>

# 2. your environment
module load anaconda3
conda activate <your_env> && export PYTHONNOUSERSITE=1

# 3. a server you can attach VS Code or a browser to, started from /jet/home
cd /jet/home/$USER
jupyter notebook --no-browser --ip=0.0.0.0
```

Then point your kernel at the printed `http://<host>:<port>/tree?token=...` URL.

Two PSC-specific things the platform cell handles for you:

- **The dataset is re-downloaded on each new node.** `$LOCAL` is node-local scratch, so it is
  empty every time you land somewhere new. The *Get the data* cell notices and re-downloads to
  `$LOCAL/imagenet100`; if you stay on the same node it finds the existing copy and skips.
- **Checkpoints go to `/jet/home`,** which persists, so resuming across jobs needs no extra
  steps - just re-run the notebook.

If `pip` cannot write into a shared read-only conda env, the install cell will say so. Either
ask for `timm` in the shared env or create your own with
`conda create -n deit --clone <your_env>`; note that `PYTHONNOUSERSITE=1` means `pip
install --user` will *not* be picked up.

## Step 2: Install / check packages

`timm` is a **hard requirement** here, not an optional extra: the transforms, Mixup, loss and
LR schedule all come from it so that the recipe matches DeiT's reference implementation. Kaggle
images already ship `torch`, `torchvision`, `timm` and `tqdm`; Colab and PSC need `timm` added
if the environment does not already have it. The cell installs only what is missing.

In [ ]:
import subprocess


def pip_install(packages):
    if not packages:
        return
    print("installing:", " ".join(packages))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=False)


required = ['timm', 'wandb', 'torchinfo']
if not IS_KAGGLE:
    required += ['kaggle']      # needed to download the dataset

def is_installed(pkg):
    try:
        return importlib.util.find_spec(pkg.replace('-', '_')) is not None
    except (ImportError, ValueError):
        return False       # odd/partial entry in sys.modules -> treat as missing


missing = [p for p in required if not is_installed(p)]
pip_install(missing)

import torch, torchvision
print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
import timm; print("timm        :", timm.__version__)   # required, see note above
print("CUDA        :", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("   ", i, torch.cuda.get_device_name(i))

## Step 3: Kaggle API credentials

- **Kaggle**: optional. Only needed to push artifacts back as a Dataset.
- **Colab / PSC / local**: **required**, because the notebook downloads ImageNet-100 for you
  through the Kaggle API.

To get a token: **Kaggle profile → Settings → Generate New Token** (new tokens start with
`KGAT_`). Treat it like a password - clear the cell before sharing the notebook.

In [ ]:
KAGGLE_USERNAME  = ""   # TODO: your Kaggle username
KAGGLE_API_TOKEN = ""   # TODO: your Kaggle token

if KAGGLE_USERNAME and KAGGLE_API_TOKEN:
    os.environ['KAGGLE_USERNAME']  = KAGGLE_USERNAME
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
    os.environ['KAGGLE_KEY']       = KAGGLE_API_TOKEN   # older kaggle clients read this name
    print("Kaggle credentials set for", KAGGLE_USERNAME)
elif IS_KAGGLE:
    print("No credentials set - fine on Kaggle, the dataset is mounted directly.")
else:
    print("[warn] No credentials set. Either fill them in so the dataset can be downloaded, "
          "or point IMAGENET100_DIR / config['data_root'] at a copy you already have.")

## Step 4: Get the data

**On Kaggle** - mount it, no download needed:

1. Right sidebar → **Input** → **Add Input**.
2. Search **`imagenet100`** and add a mirror, e.g.
   [`ambityga/imagenet100`](https://www.kaggle.com/datasets/ambityga/imagenet100).
3. Re-run the cell below; it finds the mount under `/kaggle/input` automatically.

**On Colab** - the cell downloads and unzips the dataset to `/content` (~16 GB, usually
10-20 min). It lands on the local SSD rather than Drive, because training off Drive is slow
enough to dominate epoch time. The download repeats each session; checkpoints on Drive are
what make that cheap.

**On PSC** - the cell downloads to `$LOCAL/imagenet100` (node-local scratch, typically
`/local/imagenet100`). That download repeats every time you land on a new node; staying on
the same node, it finds the existing copy and skips. Fill in the Kaggle credentials above
first. Do **not** put the dataset on `/ocean` or `/jet` - JPEG I/O from the shared filesystem
will dominate epoch time.

**Locally** - set `IMAGENET100_DIR` to an existing copy, or let it download to `./imagenet100`.

Both common on-disk layouts are recognised:

```
# layout A (ambityga/imagenet100)          # layout B (plain ImageFolder)
imagenet100/                               imagenet100/
├── train.X1/ n01968897/ *.JPEG            ├── train/ n01968897/ *.JPEG
├── train.X2/ ...                          └── val/   n01968897/ *.JPEG
├── train.X3/ ...
├── val.X/    n01968897/ *.JPEG
└── Labels.json
```

In [ ]:
def looks_like_imagenet100(path):
    path = Path(path)
    return path.is_dir() and (any(path.glob('train.X*')) or (path / 'train').is_dir())


def find_dataset(candidates, max_depth=4):
    """Breadth-first search for an ImageNet-100 tree under any of the candidates.

    Depth matters: a dataset added through Kaggle's current UI lands at
    ``/kaggle/input/datasets/<owner>/<slug>``, three levels below the mount root, while older
    notebooks see ``/kaggle/input/<slug>`` and a local unzip may add a level of its own.
    Searching breadth-first returns the shallowest match and stops before descending into the
    class folders, so this stays fast even on the full dataset.
    """
    queue = [(Path(c), 0) for c in candidates if c]
    visited = set()

    while queue:
        path, depth = queue.pop(0)
        try:
            if not path.is_dir() or path in visited:
                continue
            visited.add(path)
            if looks_like_imagenet100(path):
                return path
            if depth < max_depth:
                queue.extend((sub, depth + 1) for sub in sorted(p for p in path.iterdir() if p.is_dir()))
        except PermissionError:
            continue

    return None


def download_dataset(dest):
    print(f"Downloading {KAGGLE_DATASET} to {dest} (~16 GB, this takes a while)...")
    os.makedirs(dest, exist_ok=True)
    import kaggle
    kaggle.api.dataset_download_files(KAGGLE_DATASET, path=dest, unzip=True)
    return dest


if IS_KAGGLE:
    candidates = sorted(Path('/kaggle/input').iterdir()) if os.path.isdir('/kaggle/input') else []
elif IS_COLAB:
    candidates = [COLAB_DATA_DIR, '/content/imagenet100']
elif IS_PSC:
    candidates = [os.environ.get('IMAGENET100_DIR'),
                  f"{psc_scratch()}/imagenet100", '/local/imagenet100']
else:
    candidates = [os.environ.get('IMAGENET100_DIR'), './imagenet100', '../imagenet100']

data_root = find_dataset(candidates)

if data_root is None and not IS_KAGGLE:
    if IS_COLAB:
        target = COLAB_DATA_DIR
    elif IS_PSC:
        target = f"{psc_scratch()}/imagenet100"
    else:
        target = './imagenet100'
    download_dataset(target)
    data_root = find_dataset([target])

if data_root is None:
    raise FileNotFoundError(
        "ImageNet-100 not found.\n"
        f"  platform  : {PLATFORM}\n"
        f"  looked in : {[str(c) for c in candidates if c]}\n"
        f"  depth     : searched up to 4 levels below each of those\n"
        + ("  fix       : add the dataset via Input -> Add Input, then re-run this cell."
           if IS_KAGGLE else
           "  fix       : set KAGGLE_USERNAME/KAGGLE_API_TOKEN above to download it, "
           "or point IMAGENET100_DIR at an existing copy."
           + (" On PSC, confirm you are on a compute node so $LOCAL is set." if IS_PSC else ""))
    )

ENV['data_root'] = str(data_root)
print("Data root:", ENV['data_root'])
print("Contents :", sorted(p.name for p in Path(ENV['data_root']).iterdir())[:10])

# Imports

In [ ]:
import os
import gc
import hashlib
import io
import json
import math
import time
import copy
import random
import pickle
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.utils import make_grid
import torchvision.transforms.v2 as T
from torchvision.transforms import InterpolationMode

import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", DEVICE)

# ImageNet statistics (ImageNet-100 is a subset, so the parent statistics apply)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


torch.backends.cudnn.benchmark = True
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

# Config

### ⚠️ `Note`

- `data_root` and `checkpoint_dir` are taken from `ENV`, which the platform cell already
  resolved - you do not need to edit paths per platform. Override them here only if you want
  something non-standard.
- `batch_size` is **per optimizer micro-step**; the effective batch is
  `batch_size * grad_accum_steps`, and the learning rate is scaled from it with the linear
  rule from the DeiT paper: `lr = base_lr * eff_batch / 512`.
- `epochs`: DeiT trains for 300 epochs on full ImageNet. On a single Kaggle session you will
  get through ~60-90 epochs of ImageNet-100 at 224px, so the run is designed to be **resumed**
  across sessions (see *Checkpointing*).
- Set `smoke_test = True` for a 2-epoch / few-batch dry run before committing GPU hours.

In [ ]:
config = {
    # ---- data -------------------------------------------------------------
    'data_root'   : ENV['data_root'],      # resolved by the platform cell
    'num_classes' : 100,
    'image_size'  : 224,          # 160 or 176 trains ~2x faster; fine-tune at 224 afterwards
    'num_workers' : min(os.cpu_count() or 4, 8),
    'cache_index' : True,         # cache the (path, label) file index under ENV['work_dir']
    'precache_resize': False,     # one-time resize pass to kill the JPEG-decode bottleneck
    'precache_short_side': 256,

    # ---- augmentation: timm's exact DeiT pipeline ------------------------
    'auto_augment'       : 'rand-m9-mstd0.5-inc1',   # DeiT --aa
    'train_interpolation': 'bicubic',                # DeiT --train-interpolation
    'rrc_scale'          : (0.08, 1.0),
    're_prob'            : 0.25,                     # DeiT --reprob
    're_mode'            : 'pixel',                  # DeiT --remode
    're_count'           : 1,                        # DeiT --recount
    'eval_crop_ratio'    : 0.875,                    # DeiT --eval-crop-ratio -> resize 256
    'repeated_aug'       : True,                     # DeiT --repeated-aug (on by default)
    'ra_repeats'         : 3,                        # RASampler num_repeats

    # ---- batch-level mixing (timm Mixup) --------------------------------
    'mixup_alpha'             : 0.8,   # DeiT --mixup
    'cutmix_alpha'            : 1.0,   # DeiT --cutmix
    'mix_prob'                : 1.0,   # DeiT --mixup-prob
    'mixup_cutmix_switch_prob': 0.5,   # DeiT --mixup-switch-prob
    'mixup_mode'              : 'batch',
    'label_smoothing'         : 0.1,   # applied inside Mixup, not in the loss

    # ---- model (DeiT-S / ViT-S/16) --------------------------------------
    'model_source'   : 'timm',      # 'timm' (deit_small_patch16_224) | 'scratch' (hand-written)
    'patch_size'     : 16,
    'embed_dim'      : 384,
    'depth'          : 12,
    'num_heads'      : 6,
    'mlp_ratio'      : 4.0,
    'qkv_bias'       : True,
    'drop_rate'      : 0.0,
    'attn_drop_rate' : 0.0,
    'drop_path_rate' : 0.1,         # stochastic depth
    'distilled'      : False,       # True adds the DeiT distillation token (needs a teacher)

    # ---- optimization ----------------------------------------------------
    'epochs'          : 100,
    'warmup_epochs'   : 5,
    'batch_size'      : 128,
    'grad_accum_steps': 2,          # effective batch = 128 * 2 = 256
    'base_lr'         : 5e-4,       # DeiT reference LR for batch 512
    'base_batch'      : 512,
    'warmup_lr'       : 1e-6,       # DeiT --warmup-lr
    'min_lr'          : 1e-5,       # DeiT --min-lr
    'weight_decay'    : 0.05,
    'grad_clip'       : None,       # DeiT --clip-grad default; set 1.0 if you hit NaNs
    'ema'             : True,
    # DeiT uses 0.99996 at ~1250 steps/epoch (batch 1024 over 1.28M images), i.e. an averaging
    # horizon of 1/(1-d) = 25k steps = 7% of its 300 epochs. We have ~4x fewer steps/epoch, so
    # the same decay would average over ~half of training. 0.9999 keeps the *relative* horizon.
    'ema_decay'       : 0.9999,

    # ---- runtime ---------------------------------------------------------
    'amp'               : True,
    'use_dataparallel'  : False,    # True wraps nn.DataParallel over all visible GPUs; usually slower in a notebook
    'seed'              : 42,
    'max_runtime_hours' : 7.5 if IS_PSC else 8.5,  # leave ~30 min on an 8 h PSC interact job
    'checkpoint_dir'    : os.path.join(ENV['work_dir'], 'checkpoints'),
    'resume_from'       : None,     # e.g. "/kaggle/input/my-deit-ckpt/last.pth"
    'smoke_test'        : False,    # True -> 2 epochs, 20 batches each

    # ---- logging ---------------------------------------------------------
    'use_wandb'     : False,
    'wandb_project' : "imagenet100-deit-s",
    'run_name'      : "deit_s_scratch_in100_224",
}

config['eff_batch']  = config['batch_size'] * config['grad_accum_steps']
config['lr']         = config['base_lr'] * config['eff_batch'] / config['base_batch']
config['val_resize'] = int(round(config['image_size'] / config['eval_crop_ratio']))

# timm's Mixup in mode='batch' asserts an even batch; drop_last=True keeps it exact.
assert config['batch_size'] % 2 == 0, "timm Mixup requires an even batch size"

if config['smoke_test']:
    config.update(epochs=2, warmup_epochs=1, ema=False)

seed_everything(config['seed'])
os.makedirs(config['checkpoint_dir'], exist_ok=True)

print(json.dumps({k: str(v) for k, v in config.items()}, indent=2))
print(f"\nEffective batch: {config['eff_batch']}  ->  scaled LR: {config['lr']:.3e}")

# Dataset Discovery

ImageNet-100 ships under a few different folder conventions. `discover_imagenet100` walks the
root, finds the train/val splits, and builds a single deterministic `wnid -> class index`
mapping shared by both splits (sorted WNIDs, so the mapping is reproducible across sessions).

In [ ]:
IMG_EXTENSIONS = ('.jpeg', '.jpg', '.png', '.ppm', '.bmp', '.webp')


def _subdirs(path):
    return sorted([p for p in Path(path).iterdir() if p.is_dir()])


def discover_imagenet100(root, _depth=0):
    """Locate train/val roots for the common ImageNet-100 layouts.

    Returns:
        (train_roots, val_roots): lists of directories whose children are WNID class folders.
    """
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"data_root does not exist: {root}")

    # Layout A: train.X1 .. train.X4 + val.X
    train_roots = [p for p in sorted(root.glob('train.X*')) if p.is_dir()]
    val_roots   = [p for p in sorted(root.glob('val.X*'))   if p.is_dir()]

    # Layout B: train/ + val/
    if not train_roots and (root / 'train').is_dir():
        train_roots = [root / 'train']
    if not val_roots:
        for name in ('val', 'validation', 'valid', 'test'):
            if (root / name).is_dir():
                val_roots = [root / name]
                break

    # Layout C: everything nested one extra level deep (common with Kaggle zips)
    if not train_roots and _depth < 3:
        for sub in _subdirs(root):
            try:
                return discover_imagenet100(sub, _depth + 1)
            except FileNotFoundError:
                continue

    if not train_roots:
        raise FileNotFoundError(
            f"Could not find a train split under {root}. Contents: {[p.name for p in root.iterdir()][:20]}"
        )
    return train_roots, val_roots


def build_class_index(train_roots, val_roots):
    wnids = set()
    for r in list(train_roots) + list(val_roots):
        wnids.update(p.name for p in _subdirs(r))
    classes = sorted(wnids)
    return classes, {w: i for i, w in enumerate(classes)}


train_roots, val_roots = discover_imagenet100(config['data_root'])
classes, class_to_idx = build_class_index(train_roots, val_roots)

print("Train roots:", [str(p) for p in train_roots])
print("Val roots  :", [str(p) for p in val_roots])
print("Classes    :", len(classes), "->", classes[:5], "...")

assert len(classes) == config['num_classes'], \
    f"Found {len(classes)} classes but config['num_classes'] = {config['num_classes']}"

# Optional human-readable names. Mirrors ship Labels.json either as {wnid: name} or as a
# bare list of names; neither is required, so never let it break the run.
wnid_to_name = {}
for cand in list(Path(config['data_root']).rglob('Labels.json'))[:1]:
    try:
        with open(cand) as f:
            loaded = json.load(f)
        if isinstance(loaded, dict):
            wnid_to_name = loaded
        elif isinstance(loaded, list) and len(loaded) == len(classes):
            wnid_to_name = dict(zip(classes, loaded))
        print(f"Loaded {len(wnid_to_name)} label names from {cand}")
    except Exception as e:
        print(f"[note] ignoring {cand}: {e}")
idx_to_name = {i: wnid_to_name.get(w, w) for i, w in enumerate(classes)}

# Data Augmentations

Training uses **timm's `create_transform`** with DeiT's exact arguments, so the pipeline is the
one from `deit/datasets.py` rather than a lookalike:

| Transform | Why it is here |
|---|---|
| `RandomResizedCropAndInterpolation(scale=(0.08, 1.0), bicubic)` | scale/aspect invariance; the strongest single augmentation |
| `RandomHorizontalFlip` | objects are left-right symmetric for classification |
| `RandAugment` via `rand-m9-mstd0.5-inc1` | DeiT's policy: 2 ops, magnitude 9, magnitude noise std 0.5, *increasing*-severity op variants |
| `Normalize(ImageNet stats)` | zero-centred inputs, stable LayerNorm statistics |
| `RandomErasing(p=0.25, mode='pixel')` | occlusion robustness; discourages reliance on single patches |
| MixUp / CutMix *(next section)* | batch-level soft labels; critical ViT regularization |

Two details worth knowing. `rand-m9-mstd0.5-inc1` is **not** the same as torchvision's
`RandAugment(num_ops=2, magnitude=9)`: the `inc1` suffix selects op variants whose strength
increases with magnitude, and `mstd0.5` jitters the magnitude per call. And DeiT passes
`color_jitter=0.3` alongside `auto_augment`, but timm **ignores colour jitter whenever an
auto-augment policy is set** - so that argument is dead config in the reference and is omitted
here.

Validation is DeiT's deterministic `Resize(image_size / 0.875) → CenterCrop(image_size)` with
bicubic interpolation, i.e. `Resize(256) → CenterCrop(224)`.

In [ ]:
from timm.data import create_transform


def create_transforms(cfg, is_training):
    """DeiT's transform pipeline.

    Training is delegated to timm's ``create_transform`` with the arguments from
    ``deit/datasets.py``. Evaluation is DeiT's deterministic resize/center-crop, written out
    explicitly because it is only three lines and keeps the normalization visible.
    """
    if is_training:
        return create_transform(
            input_size=cfg['image_size'],
            is_training=True,
            scale=cfg['rrc_scale'],
            auto_augment=cfg['auto_augment'],
            interpolation=cfg['train_interpolation'],
            re_prob=cfg['re_prob'],
            re_mode=cfg['re_mode'],
            re_count=cfg['re_count'],
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD,
        )

    return T.Compose([
        T.Resize(cfg['val_resize'], interpolation=InterpolationMode.BICUBIC, antialias=True),
        T.CenterCrop(cfg['image_size']),
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


train_transforms = create_transforms(config, is_training=True)
val_transforms   = create_transforms(config, is_training=False)

print("TRAIN:", train_transforms)
print("\nVAL  :", val_transforms)

# Datasets and DataLoaders

In [ ]:
class ImageNet100Dataset(Dataset):
    """ImageFolder-style dataset that can span several root directories.

    ImageNet-100 is often distributed as four shards (``train.X1`` .. ``train.X4``) that each
    hold a subset of the class folders, so a single-root ``ImageFolder`` is not enough. The
    class -> index mapping is injected rather than inferred, which guarantees train and val
    agree even when a shard is missing a class.
    """

    def __init__(self, roots, class_to_idx, transform, cache_path=None):
        self.roots = [Path(r) for r in (roots if isinstance(roots, (list, tuple)) else [roots])]
        self.class_to_idx = class_to_idx
        self.transform = transform
        self.samples = self._load_index(cache_path)
        if len(self.samples) == 0:
            raise RuntimeError(f"No images found under {[str(r) for r in self.roots]}")
        self.targets = [lbl for _, lbl in self.samples]
        self.classes = sorted(class_to_idx, key=class_to_idx.get)

    # ---------------- index construction ----------------
    def _scan(self):
        samples = []
        for root in self.roots:
            for class_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
                wnid = class_dir.name
                if wnid not in self.class_to_idx:
                    continue
                label = self.class_to_idx[wnid]
                for fname in sorted(os.listdir(class_dir)):
                    if fname.lower().endswith(IMG_EXTENSIONS):
                        samples.append((str(class_dir / fname), label))
        return samples

    def _load_index(self, cache_path):
        if cache_path and os.path.exists(cache_path):
            try:
                with open(cache_path, 'rb') as f:
                    return pickle.load(f)
            except Exception:
                pass
        samples = self._scan()
        if cache_path:
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            with open(cache_path, 'wb') as f:
                pickle.dump(samples, f)
        return samples

    # ---------------- torch Dataset API ----------------
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with open(path, 'rb') as f:
            img = Image.open(io.BytesIO(f.read())).convert('RGB')
        return self.transform(img), label

## Repeated Augmentation (DeiT's `RASampler`)

DeiT turns this on by default (`--repeated-aug`, `parser.set_defaults(repeated_aug=True)`). It
draws each image **3 times per epoch with independent augmentations**, truncating the epoch to
`len(dataset) // 256 * 256` samples - so the number of images seen per epoch is unchanged, but
only about a third of them are *distinct*, each appearing three times.

One honest caveat about porting it here. The reference `RASampler` is a `DistributedSampler`
variant, and `main.py` only uses it on the distributed path - a single-process run falls back to
`RandomSampler`. With 8 ranks the stride `indices[rank::num_replicas]` scatters an image's three
copies across three different GPUs, so repeats land in the *global* batch rather than in one
rank's batch. Single-process, the copies end up adjacent, which reproduces the same global
behaviour at our effective batch of 256 (three views of ~85 distinct images) but makes the
gradient within a micro-batch more correlated than DeiT's.

Set `config['repeated_aug'] = False` to fall back to plain shuffling if you would rather spend
the epoch on distinct images.

In [ ]:
class RASampler(torch.utils.data.Sampler):
    """Repeated-augmentation sampler, the num_replicas=1 case of deit/samplers.py."""

    def __init__(self, dataset, num_repeats=3, shuffle=True, seed=0):
        if num_repeats < 1:
            raise ValueError("num_repeats should be greater than 0")
        self.num_samples = len(dataset)
        self.num_repeats = num_repeats
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        # DeiT truncates the epoch to a multiple of 256; guard the tiny-dataset case where
        # that would floor to zero and hand the DataLoader an empty epoch.
        self.num_selected_samples = (self.num_samples // 256) * 256 or self.num_samples

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        if self.shuffle:
            g = torch.Generator()
            g.manual_seed(self.seed + self.epoch)
            indices = torch.randperm(self.num_samples, generator=g)
        else:
            indices = torch.arange(self.num_samples)

        indices = torch.repeat_interleave(indices, repeats=self.num_repeats, dim=0)
        return iter(indices[:self.num_selected_samples].tolist())

    def __len__(self):
        return self.num_selected_samples

## Optional: pre-resize the images once

This does not touch the recipe - it removes an I/O bottleneck. Every epoch currently decodes
full-resolution ImageNet JPEGs (often 500x375 or larger) and throws most of the pixels away in
`RandomResizedCrop`. With 4 vCPUs on Kaggle that decode, not the GPU, sets your epoch time -
and repeated augmentation makes it **3x worse**, since each image is now decoded three times
per epoch.

Setting `config['precache_resize'] = True` does one pass that resizes every image's short side
to 256px into a scratch folder, then points the dataset at it. Expect ~10-15 minutes once, and
meaningfully faster epochs after that. The 256px short side preserves the eval pipeline exactly
(`Resize(256) → CenterCrop(224)`) and leaves `RandomResizedCrop` with ample pixels.

The file index is cached per resolved root, so switching this flag on or off automatically
rebuilds the index instead of silently reusing a stale one.

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def _resize_one(job):
    src_path, dst_path, short_side = job
    if os.path.exists(dst_path):
        return 0
    try:
        with Image.open(src_path) as im:
            im = im.convert('RGB')
            w, h = im.size
            scale = short_side / min(w, h)
            if scale < 1.0:
                im = im.resize((max(short_side, round(w * scale)),
                                max(short_side, round(h * scale))), Image.BICUBIC)
            im.save(dst_path, 'JPEG', quality=90)
        return 1
    except Exception:
        return -1


def precache_resized(roots, out_dir, short_side=256, workers=None):
    """Resize every image's short side once into out_dir, mirroring the folder layout."""
    out_dir = Path(out_dir)
    jobs = []
    for root in roots:
        root = Path(root)
        for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            dst_dir = out_dir / root.name / class_dir.name
            dst_dir.mkdir(parents=True, exist_ok=True)
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(IMG_EXTENSIONS):
                    jobs.append((str(class_dir / fname), str(dst_dir / fname), short_side))

    # One marker per root set: train and val share out_dir but have different job counts.
    key = hashlib.md5('|'.join(sorted(str(r) for r in roots)).encode()).hexdigest()[:8]
    marker = out_dir / f'.done-{key}'
    if marker.exists() and marker.read_text().strip() == str(len(jobs)):
        print(f"Re-using resized cache at {out_dir} ({len(jobs):,} images)")
    else:
        # Threads, not processes: PIL releases the GIL inside decode/resize/encode, and a
        # thread pool needs no pickling, which a function defined in a notebook cell cannot
        # always support.
        workers = workers or max(1, 2 * (os.cpu_count() or 4))
        print(f"Resizing {len(jobs):,} images to short side {short_side} "
              f"with {workers} threads -> {out_dir}")
        failed = 0
        with ThreadPoolExecutor(max_workers=workers) as ex:
            for r in tqdm(ex.map(_resize_one, jobs, chunksize=64),
                          total=len(jobs), desc="resize", dynamic_ncols=True):
                failed += (r == -1)
        if failed:
            print(f"[warn] {failed} images could not be converted and were skipped")
        marker.write_text(str(len(jobs)))

    return [out_dir / Path(r).name for r in roots]


if config['precache_resize']:
    scratch = os.path.join(ENV['work_dir'], f"resized{config['precache_short_side']}")
    train_roots = precache_resized(train_roots, scratch, config['precache_short_side'])
    val_roots   = precache_resized(val_roots,   scratch, config['precache_short_side'])
    print("Train roots now:", [str(p) for p in train_roots])
else:
    print("precache_resize is off - decoding original JPEGs every epoch.")

In [ ]:
gc.collect()

cache_dir = os.path.join(ENV['work_dir'], 'cache')

def index_cache_path(split, roots):
    """Cache file keyed by the resolved roots, so toggling precache_resize rebuilds it."""
    if not config['cache_index']:
        return None
    key = hashlib.md5('|'.join(str(r) for r in roots).encode()).hexdigest()[:8]
    return f"{cache_dir}/{split}_index_{key}.pkl"


train_dataset = ImageNet100Dataset(
    roots=train_roots,
    class_to_idx=class_to_idx,
    transform=train_transforms,
    cache_path=index_cache_path('train', train_roots),
)
val_dataset = ImageNet100Dataset(
    roots=val_roots,
    class_to_idx=class_to_idx,
    transform=val_transforms,
    cache_path=index_cache_path('val', val_roots),
)

ra_sampler = RASampler(train_dataset, num_repeats=config['ra_repeats']) \
    if config['repeated_aug'] else None

train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    sampler=ra_sampler,                       # None -> fall back to shuffle=True
    shuffle=(ra_sampler is None),
    num_workers=config['num_workers'],
    pin_memory=True,
    drop_last=True,                 # keeps the Mixup batch size even
    persistent_workers=config['num_workers'] > 0,
    prefetch_factor=4 if config['num_workers'] > 0 else None,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'] * 2,
    shuffle=False,
    num_workers=config['num_workers'],
    pin_memory=True,
    persistent_workers=config['num_workers'] > 0,
)

if ra_sampler is not None:
    unique = ra_sampler.num_selected_samples / config['ra_repeats']
    print(f"Repeated augmentation: {config['ra_repeats']}x -> "
          f"{ra_sampler.num_selected_samples:,} samples/epoch from ~{unique:,.0f} distinct images")

print(f"Train images : {len(train_dataset):,}  ({len(train_loader):,} batches)")
print(f"Val images   : {len(val_dataset):,}  ({len(val_loader):,} batches)")
print(f"Image shape  : {tuple(train_dataset[0][0].shape)}")

## MixUp / CutMix

These operate on an assembled **batch**, so they run after the `DataLoader` (on GPU, which is
free). This is **timm's `Mixup`** with DeiT's arguments, which is more than a convenience:

- it switches between MixUp and CutMix per batch with probability `switch_prob=0.5`,
- it **folds label smoothing into the targets** (`label_smoothing=0.1`), which is why the
  training loss below is `SoftTargetCrossEntropy` with *no* smoothing of its own - smoothing it
  again would double-apply it,
- it always returns soft targets `[B, num_classes]`, even on batches where mixing did not fire.

`mode='batch'` mixes the batch against its own reversal and **asserts an even batch size**, so
`batch_size` must be even and the train loader uses `drop_last=True`.

In [ ]:
from timm.data import Mixup

mixup_fn = Mixup(
    mixup_alpha=config['mixup_alpha'],
    cutmix_alpha=config['cutmix_alpha'],
    cutmix_minmax=None,
    prob=config['mix_prob'],
    switch_prob=config['mixup_cutmix_switch_prob'],
    mode=config['mixup_mode'],
    label_smoothing=config['label_smoothing'],
    num_classes=config['num_classes'],
)

mixup_active = config['mixup_alpha'] > 0 or config['cutmix_alpha'] > 0


def apply_batch_mix(images, labels):
    """MixUp/CutMix a batch, returning soft targets. Mirrors DeiT's train_one_epoch."""
    if not mixup_active:
        return images, labels
    return mixup_fn(images, labels)


print("Mixup active:", mixup_active, "| targets become soft [B, num_classes]")

# EDA and Viz

In [ ]:
counts = pd.Series(train_dataset.targets).value_counts().sort_index()

print("Classes              :", len(classes))
print("Train images         :", f"{len(train_dataset):,}")
print("Val images           :", f"{len(val_dataset):,}")
print("Images/class (train) :", f"min {counts.min()}, median {int(counts.median())}, max {counts.max()}")
print("Batch size           :", config['batch_size'], f"(effective {config['eff_batch']})")

fig, ax = plt.subplots(figsize=(10, 2.6))
ax.bar(counts.index, counts.values, width=1.0)
ax.set_xlabel("class index"); ax.set_ylabel("# train images")
ax.set_title("ImageNet-100 train distribution")
plt.tight_layout(); plt.show()

In [ ]:
def denormalize(x):
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1)


def show_dataset_samples(loader, title, n=8, mix=False):
    images, labels = next(iter(loader))
    images, labels = images[:n], labels[:n]
    if mix:
        images, soft = apply_batch_mix(images, labels)
        labels = soft.argmax(1) if soft.ndim == 2 else soft

    grid = make_grid(denormalize(images), nrow=n, padding=2)
    fig, ax = plt.subplots(figsize=(2 * n, 2.8))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.axis('off')
    ax.set_title(f"{title}\n" + " | ".join(str(idx_to_name[int(l)])[:14] for l in labels), fontsize=8)
    plt.tight_layout(); plt.show()


show_dataset_samples(train_loader, "Train samples (RandAugment + RandomErasing)")
show_dataset_samples(val_loader,   "Validation samples (center crop)")
show_dataset_samples(train_loader, "Train samples after MixUp / CutMix", mix=True)

# Model Architecture

## DeiT-S in one paragraph

A `224x224` image is cut into `16x16` patches by a single strided convolution, giving
`(224/16)^2 = 196` tokens of width 384. A learnable **class token** is prepended and a learnable
**positional embedding** is added, because self-attention is permutation invariant and would
otherwise be blind to where a patch came from. The 197 tokens pass through 12 identical
pre-norm transformer blocks:

```
x = x + DropPath(Attention(LayerNorm(x)))
x = x + DropPath(MLP(LayerNorm(x)))
```

Multi-head self-attention (6 heads x 64 dims) lets every patch read every other patch from
layer 1 onward - the global receptive field that CNNs only reach after many layers. The MLP
(expansion 4, GELU) mixes channels within each token. After a final LayerNorm, the class token
is fed to a linear classifier.

## DeiT vs. ViT

The architecture *is* ViT. DeiT's contribution is (a) the training recipe - heavy augmentation,
AdamW with cosine decay, stochastic depth, repeated MixUp/CutMix, EMA - and (b) an optional
**distillation token** that learns from a CNN teacher. Since we train from scratch without a
teacher, `config['distilled'] = False`, i.e. this is DeiT-S without distillation (the
`deit_small_patch16_224` model of the paper, 79.8% on full ImageNet).

## Design notes in the code below

- **`DropPath` (stochastic depth)** drops whole residual branches per sample with a rate that
  increases linearly with depth. It is the main depth regularizer for ViTs.
- **Attention** uses `F.scaled_dot_product_attention` when available (fused FlashAttention
  kernel - noticeably faster and lighter on memory) with an explicit maths fallback so the
  implementation stays readable and portable.
- **Initialization** follows the paper: truncated normal (std 0.02) for the positional
  embedding, class token and all linear weights; zeros for biases and LayerNorm shifts.
- **`no_weight_decay`** marks parameters that must be excluded from weight decay - decaying
  the positional embedding or class token measurably hurts.
- The forward pass returns a **dict** `{'out': logits, 'feats': cls_embedding}` so downstream
  code (linear probing, retrieval, distillation) can reuse the representation.

In [ ]:
def trunc_normal_(tensor, std=0.02):
    return nn.init.trunc_normal_(tensor, mean=0.0, std=std, a=-2 * std, b=2 * std)


class DropPath(nn.Module):
    """Stochastic depth, applied per sample on the residual branch."""

    def __init__(self, p: float = 0.0):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.p == 0.0 or not self.training:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
        return x * mask / keep

    def extra_repr(self):
        return f"p={self.p:.3f}"


class PatchEmbed(nn.Module):
    """Image -> sequence of patch tokens via a non-overlapping strided conv."""

    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=384):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        assert H == self.img_size and W == self.img_size, \
            f"Input {H}x{W} does not match model img_size {self.img_size}"
        x = self.proj(x)                 # B, D, H/P, W/P
        return x.flatten(2).transpose(1, 2)   # B, N, D


class Attention(nn.Module):
    """Multi-head self-attention."""

    def __init__(self, dim, num_heads=6, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.attn_drop_p = attn_drop

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self._fused = hasattr(F, 'scaled_dot_product_attention')

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]      # each B, heads, N, head_dim

        if self._fused:
            x = F.scaled_dot_product_attention(
                q, k, v, dropout_p=self.attn_drop_p if self.training else 0.0
            )
        else:
            attn = (q @ k.transpose(-2, -1)) * self.scale
            attn = self.attn_drop(attn.softmax(dim=-1))
            x = attn @ v

        x = x.transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))


class Mlp(nn.Module):
    def __init__(self, dim, hidden_dim, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class Block(nn.Module):
    """Pre-norm transformer block: attention then MLP, both residual."""

    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=True,
                 drop=0.0, attn_drop=0.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads, qkv_bias, attn_drop, drop)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, int(dim * mlp_ratio), drop)
        self.drop_path = DropPath(drop_path)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class DeiT(nn.Module):
    """Data-efficient image transformer (DeiT), optionally with a distillation token."""

    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=100,
                 embed_dim=384, depth=12, num_heads=6, mlp_ratio=4.0, qkv_bias=True,
                 drop_rate=0.0, attn_drop_rate=0.0, drop_path_rate=0.1, distilled=False):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.distilled = distilled
        self.num_tokens = 2 if distilled else 1

        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim)) if distilled else None
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + self.num_tokens, embed_dim))
        self.pos_drop = nn.Dropout(drop_rate)

        # linearly increasing stochastic depth, as in the paper
        dpr = torch.linspace(0, drop_path_rate, depth).tolist()
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, qkv_bias,
                  drop_rate, attn_drop_rate, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)

        self.head = nn.Linear(embed_dim, num_classes)
        self.head_dist = nn.Linear(embed_dim, num_classes) if distilled else None

        self.apply(self._init_weights)
        trunc_normal_(self.pos_embed, std=0.02)
        trunc_normal_(self.cls_token, std=0.02)
        if self.dist_token is not None:
            trunc_normal_(self.dist_token, std=0.02)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        # the patch-embedding Conv2d keeps PyTorch's default init, as in the official DeiT code

    def no_weight_decay(self):
        return {'pos_embed', 'cls_token', 'dist_token'}

    def forward_features(self, x):
        x = self.patch_embed(x)
        B = x.shape[0]

        tokens = [self.cls_token.expand(B, -1, -1)]
        if self.dist_token is not None:
            tokens.append(self.dist_token.expand(B, -1, -1))
        x = torch.cat(tokens + [x], dim=1)

        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

    def forward(self, x):
        x = self.forward_features(x)
        cls = x[:, 0]
        out = {'feats': cls}

        if self.distilled:
            logits_cls = self.head(cls)
            logits_dist = self.head_dist(x[:, 1])
            out['logits_cls'] = logits_cls
            out['logits_dist'] = logits_dist
            out['out'] = (logits_cls + logits_dist) / 2
        else:
            out['out'] = self.head(cls)
        return out


def deit_small_patch16(num_classes=100, img_size=224, **kw):
    """DeiT-S / ViT-S/16: dim 384, depth 12, 6 heads (~21.7M params)."""
    defaults = dict(patch_size=16, embed_dim=384, depth=12, num_heads=6, mlp_ratio=4.0)
    defaults.update(kw)
    return DeiT(img_size=img_size, num_classes=num_classes, **defaults)


def forward_logits(model, images):
    """Models here return a dict; timm models return a tensor. Normalize both."""
    out = model(images)
    return out['out'] if isinstance(out, dict) else out


def unwrap(model):
    return model.module if isinstance(model, (nn.DataParallel, nn.parallel.DistributedDataParallel)) else model

## Build the model

`model_source='timm'` swaps in `timm.create_model('deit_small_patch16_224', pretrained=False)`.
It is the *same* architecture with *random* weights, which is useful for verifying the
from-scratch implementation reaches comparable accuracy and throughput.

In [ ]:
if config['model_source'] == 'timm':
    import timm
    model = timm.create_model(
        'deit_small_patch16_224',
        pretrained=False,                      # train from scratch
        num_classes=config['num_classes'],
        img_size=config['image_size'],
        drop_rate=config['drop_rate'],
        drop_path_rate=config['drop_path_rate'],
    )
else:
    model = deit_small_patch16(
        num_classes=config['num_classes'],
        img_size=config['image_size'],
        patch_size=config['patch_size'],
        embed_dim=config['embed_dim'],
        depth=config['depth'],
        num_heads=config['num_heads'],
        mlp_ratio=config['mlp_ratio'],
        qkv_bias=config['qkv_bias'],
        drop_rate=config['drop_rate'],
        attn_drop_rate=config['attn_drop_rate'],
        drop_path_rate=config['drop_path_rate'],
        distilled=config['distilled'],
    )

model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model source : {config['model_source']}")
print(f"Params       : {n_params:,} ({n_params / 1e6:.2f} M)")

# Shape / throughput sanity check
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, config['image_size'], config['image_size'], device=DEVICE)
    logits = forward_logits(model, dummy)
print(f"Forward check: {tuple(dummy.shape)} -> {tuple(logits.shape)}")

try:
    from torchinfo import summary
    summary(model, input_size=(1, 3, config['image_size'], config['image_size']), depth=2)
except Exception as e:
    print("torchinfo summary skipped:", e)

In [ ]:
# DataParallel is off by default. It can use a second Kaggle T4, but the scatter/gather
# overhead often makes a notebook run *slower* than a single GPU. Turn it on only if you
# have measured a speedup. For a serious multi-GPU run, use DDP from a script instead.
if config['use_dataparallel'] and DEVICE == 'cuda' and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Wrapped in DataParallel over {torch.cuda.device_count()} GPUs")
else:
    print(f"Single-GPU training ({torch.cuda.device_count()} device(s) visible)")

gc.collect()
torch.cuda.empty_cache()

# Loss | Optimizer | Scheduler

**Loss** - `SoftTargetCrossEntropy` for training, because timm's Mixup already emits smoothed
soft targets. Validation uses a plain `nn.CrossEntropyLoss` on hard labels, exactly as DeiT's
`engine.evaluate` does. These are **not interchangeable**: `SoftTargetCrossEntropy` raises a
shape error on integer labels, so every `validate(...)` call must be passed `val_criterion`.

**Optimizer** - AdamW. SGD with momentum, the CNN default, trains ViTs poorly; DeiT uses AdamW
with `weight_decay=0.05`. Parameters are split into two groups so **biases, LayerNorm weights,
the class token and the positional embedding receive no weight decay** - these are not
redundant-capacity parameters and decaying them hurts.

**Schedule** - timm's `CosineLRScheduler` via `create_scheduler_v2`, stepped **once per epoch**
after the epoch completes, which is what `deit/main.py` does. Linear warm-up runs for 5 epochs
from `warmup_lr=1e-6` (ViTs diverge without it: attention logits blow up while the positional
embedding is still random), then cosine decay to `min_lr=1e-5` at epoch `epochs`.

`cooldown_epochs=0` deliberately: DeiT's `--cooldown-epochs` default is 10, but it discards the
epoch count that `create_scheduler` returns and loops exactly `args.epochs` times, so the
cooldown is never reached. It also leaves `t_initial` untouched, so passing 0 reproduces DeiT's
LR curve identically (verified: the trajectories are equal).

**AMP** - `float16` autocast plus `GradScaler`. Roughly 2x faster and halves activation memory
on T4/P100, which is what makes 224px DeiT-S trainable in a notebook session.

In [ ]:
from timm.loss import SoftTargetCrossEntropy, LabelSmoothingCrossEntropy

# Training loss must match how the targets were built (see the Mixup cell).
if mixup_active:
    criterion = SoftTargetCrossEntropy()
else:
    criterion = LabelSmoothingCrossEntropy(smoothing=config['label_smoothing'])

# Validation always sees hard integer labels.
val_criterion = nn.CrossEntropyLoss()

print("train criterion:", type(criterion).__name__, "| val criterion:", type(val_criterion).__name__)

# ---- parameter groups: no weight decay on norms, biases, tokens, pos-embed ----
base_model = unwrap(model)
skip_keywords = set()
if hasattr(base_model, 'no_weight_decay'):
    skip_keywords = set(base_model.no_weight_decay())

decay, no_decay, no_decay_names = [], [], []
for name, p in base_model.named_parameters():
    if not p.requires_grad:
        continue
    if p.ndim <= 1 or name.endswith('.bias') or any(k in name for k in skip_keywords):
        no_decay.append(p)
        no_decay_names.append(name)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW(
    [
        {'params': decay,    'weight_decay': config['weight_decay']},
        {'params': no_decay, 'weight_decay': 0.0},
    ],
    lr=config['lr'],
    betas=(0.9, 0.999),
    eps=1e-8,
)

print(f"Decayed tensors    : {len(decay)}")
print(f"Non-decayed tensors: {len(no_decay)}  e.g. {no_decay_names[:4]}")

# ---- timm's cosine schedule, stepped once per epoch (as in deit/main.py) ----
from timm.scheduler import create_scheduler_v2

scheduler, _ = create_scheduler_v2(
    optimizer,
    sched='cosine',
    num_epochs=config['epochs'],
    min_lr=config['min_lr'],
    warmup_lr=config['warmup_lr'],
    warmup_epochs=config['warmup_epochs'],
    cooldown_epochs=0,
)

amp_on = config['amp'] and DEVICE == 'cuda'
try:
    scaler = torch.amp.GradScaler('cuda', enabled=amp_on)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=amp_on)

steps_per_epoch = math.ceil(len(train_loader) / config['grad_accum_steps'])
print(f"Scheduler: {type(scheduler).__name__} | {config['epochs']} epochs "
      f"({config['warmup_epochs']} warm-up) | {steps_per_epoch} optimizer steps/epoch")
print(f"EMA horizon: ~{1 / (1 - config['ema_decay']):.0f} steps "
      f"= {1 / (1 - config['ema_decay']) / steps_per_epoch:.1f} epochs")


def schedule_preview(num_epochs):
    """LR the scheduler will use at the start of each epoch."""
    try:
        return [scheduler._get_lr(e)[0] for e in range(num_epochs)]
    except Exception:          # fall back to simulating a throwaway scheduler
        probe = torch.optim.AdamW([torch.zeros(1, requires_grad=True)], lr=config['lr'])
        sch, _ = create_scheduler_v2(probe, sched='cosine', num_epochs=num_epochs,
                                     min_lr=config['min_lr'], warmup_lr=config['warmup_lr'],
                                     warmup_epochs=config['warmup_epochs'], cooldown_epochs=0)
        out = []
        for e in range(num_epochs):
            out.append(probe.param_groups[0]['lr'])
            sch.step(e + 1)
        return out


lrs = schedule_preview(config['epochs'])
plt.figure(figsize=(7, 2.4))
plt.plot(range(1, config['epochs'] + 1), lrs)
plt.axvline(config['warmup_epochs'], color='r', ls='--', lw=1, label='end of warm-up')
plt.xlabel("epoch"); plt.ylabel("lr"); plt.title("LR schedule (timm cosine)")
plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Exponential Moving Average of weights

An EMA shadow copy of the weights is a nearly free accuracy boost (typically +0.3-1.0 top-1)
for transformer training, because the cosine tail still bounces around a wide minimum. Both the
raw and the EMA model are evaluated each epoch and checkpointed independently.

In [ ]:
class ModelEma:
    """Exponential moving average of model parameters and buffers."""

    def __init__(self, model, decay=0.9998, warmup=True):
        self.module = copy.deepcopy(unwrap(model)).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.warmup = warmup
        self.updates = 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = self.decay
        if self.warmup:  # ramp the decay in so early steps are not dominated by the init
            d = min(d, (1 + self.updates) / (10 + self.updates))
        msd = unwrap(model).state_dict()
        for k, v in self.module.state_dict().items():
            src = msd[k].detach()
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src.to(v.device), alpha=1.0 - d)
            else:
                v.copy_(src.to(v.device))

    def state_dict(self):
        return self.module.state_dict()

    def load_state_dict(self, sd):
        self.module.load_state_dict(sd)


model_ema = ModelEma(model, decay=config['ema_decay']) if config['ema'] else None
print("EMA:", "enabled" if model_ema else "disabled")

# Metrics

In [ ]:
class AverageMeter:
    """Tracks the running average of a scalar metric."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = self.avg = self.sum = 0.0
        self.count = 0

    def update(self, val, n: int = 1):
        self.val = float(val)
        self.sum += float(val) * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


def topk_accuracy(logits, targets, topk=(1, 5)):
    """Top-k accuracy in percent. Soft (MixUp) targets are reduced with argmax."""
    if targets.ndim == 2:
        targets = targets.argmax(dim=1)

    maxk = min(max(topk), logits.size(1))
    batch_size = targets.size(0)

    _, preds = logits.topk(maxk, dim=1, largest=True, sorted=True)
    correct = preds.t().eq(targets.view(1, -1))

    out = []
    for k in topk:
        k = min(k, maxk)
        out.append(correct[:k].reshape(-1).float().sum(0) * 100.0 / batch_size)
    return out


def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"

# Train and Validation Functions

`train_epoch` combines the pieces that make ViT training work on a single notebook GPU:

- **gradient accumulation** so the effective batch (256) is larger than what fits in memory,
- **AMP** with loss scaling,
- **gradient clipping** after `scaler.unscale_` (clipping scaled gradients would be wrong),
- **per-step EMA updates**, tied to *optimizer* steps rather than batches (the LR schedule
  steps once per epoch instead, in the training loop, as DeiT does),
- accuracy measured against the **original hard labels** even when MixUp replaced them, so the
  number stays interpretable.

In [ ]:
def train_epoch(model, dataloader, optimizer, scaler, device, criterion,
                ema=None, accum_steps=1, grad_clip=None, epoch=0, limit_batches=None):
    """One training epoch. Returns (top1, top5, loss)."""
    model.train()
    loss_meter, top1_meter, top5_meter = AverageMeter(), AverageMeter(), AverageMeter()

    n_batches = len(dataloader) if limit_batches is None else min(limit_batches, len(dataloader))
    progress = tqdm(dataloader, total=n_batches, desc=f"Train {epoch + 1}",
                    dynamic_ncols=True, leave=False)
    optimizer.zero_grad(set_to_none=True)
    amp_enabled = scaler.is_enabled()

    for i, (images, labels) in enumerate(progress):
        if i >= n_batches:
            break

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        hard_labels = labels                       # kept for the accuracy metric
        images, labels = apply_batch_mix(images, labels)

        with torch.amp.autocast('cuda', enabled=amp_enabled):
            logits = forward_logits(model, images)
            loss = criterion(logits, labels)

        scaler.scale(loss / accum_steps).backward()

        is_step = ((i + 1) % accum_steps == 0) or (i + 1 == n_batches)
        if is_step:
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)

        batch_loss = loss.item()
        if not math.isfinite(batch_loss):
            print(f"  [warn] non-finite loss at batch {i}; skipping its statistics")
        else:
            acc1, acc5 = topk_accuracy(logits.detach().float(), hard_labels, topk=(1, 5))
            bs = images.size(0)
            loss_meter.update(batch_loss, bs)
            top1_meter.update(acc1.item(), bs)
            top5_meter.update(acc5.item(), bs)

        progress.set_postfix(
            loss=f"{loss_meter.avg:.4f}",
            top1=f"{top1_meter.avg:.2f}%",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}",
        )

    return top1_meter.avg, top5_meter.avg, loss_meter.avg


@torch.no_grad()
def validate(model, dataloader, device, criterion, amp=True, desc="Val", return_preds=False):
    """Evaluate top-1 / top-5 / loss. Optionally return predictions and targets."""
    model.eval()
    loss_meter, top1_meter, top5_meter = AverageMeter(), AverageMeter(), AverageMeter()
    all_preds, all_targets = [], []

    progress = tqdm(dataloader, desc=desc, dynamic_ncols=True, leave=False)
    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=amp and device == 'cuda'):
            logits = forward_logits(model, images)
        logits = logits.float()
        loss = criterion(logits, labels)

        acc1, acc5 = topk_accuracy(logits, labels, topk=(1, 5))
        bs = images.size(0)
        loss_meter.update(loss.item(), bs)
        top1_meter.update(acc1.item(), bs)
        top5_meter.update(acc5.item(), bs)

        if return_preds:
            all_preds.append(logits.argmax(1).cpu())
            all_targets.append(labels.cpu())

        progress.set_postfix(loss=f"{loss_meter.avg:.4f}", top1=f"{top1_meter.avg:.2f}%")

    if return_preds:
        return (top1_meter.avg, top5_meter.avg, loss_meter.avg,
                torch.cat(all_preds).numpy(), torch.cat(all_targets).numpy())
    return top1_meter.avg, top5_meter.avg, loss_meter.avg

# WandB (optional)

Set `config['use_wandb'] = True` and paste your key. Everything still works with it off - the
history is also kept locally in `history.json` and plotted at the end. Checkpoints are mirrored
to the run with `wandb.save` as they are written.

In [ ]:
WANDB_KEY = ""   # TODO: paste your key from wandb.ai/settings (leave empty to skip)

run = None
if config['use_wandb'] and WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    run = wandb.init(project=config['wandb_project'], name=config['run_name'], config=config)
    print("WandB run:", run.url)
else:
    print("WandB disabled - metrics will be stored in history.json")

# Checkpointing and Resuming

Both Kaggle and Colab sessions are time limited, and DeiT-S needs more epochs than one session
provides. The loop therefore writes `last.pth` **every epoch** into `ENV['work_dir']` and stops
cleanly once `config['max_runtime_hours']` is reached.

Resuming is automatic as long as the checkpoint is still reachable, which is exactly why
`work_dir` is platform-aware:

- **Colab**: `work_dir` is on Google Drive, so `last.pth` is already there in the next session.
  Just re-run top to bottom - it finds the checkpoint with no extra steps.
- **Kaggle**: `/kaggle/working` is wiped between sessions, so after the run stops either
  *Save Version*, or create a Dataset from `checkpoints/last.pth`. In the next session add it
  as an Input and set `config['resume_from'] = "/kaggle/input/<your-dataset>/last.pth"`.

- **PSC**: `work_dir` is on `/jet/home`, which persists across jobs and nodes, so re-running
  the notebook resumes by itself. Only the dataset on `$LOCAL` needs re-downloading.
- **Local**: nothing to do, `./work` persists.

Optimizer, scheduler, scaler, EMA and epoch counter are all restored, so the cosine schedule
continues rather than restarting.

In [ ]:
CKPT_DIR = config['checkpoint_dir']
os.makedirs(CKPT_DIR, exist_ok=True)


def save_checkpoint(path, model, optimizer, scheduler, scaler, ema, epoch, metrics, history):
    torch.save({
        'model_state_dict'    : unwrap(model).state_dict(),
        'ema_state_dict'      : ema.state_dict() if ema is not None else None,
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict'   : scaler.state_dict() if scaler is not None else None,
        'epoch'               : epoch,
        'metrics'             : metrics,
        'history'             : history,
        'config'              : config,
        'classes'             : classes,
    }, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None, ema=None,
                    device='cuda', strict=True):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    missing = unwrap(model).load_state_dict(ckpt['model_state_dict'], strict=strict)
    if ema is not None and ckpt.get('ema_state_dict') is not None:
        ema.load_state_dict(ckpt['ema_state_dict'])
    if optimizer is not None and ckpt.get('optimizer_state_dict'):
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if scheduler is not None and ckpt.get('scheduler_state_dict'):
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if scaler is not None and ckpt.get('scaler_state_dict'):
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    print(f"Loaded {path} | epoch {ckpt.get('epoch', -1) + 1} | metrics: {ckpt.get('metrics')}")
    return ckpt


start_epoch = 0
best_top1 = 0.0
history = {'epoch': [], 'lr': [], 'train_loss': [], 'train_top1': [],
           'val_loss': [], 'val_top1': [], 'val_top5': [], 'ema_top1': []}

resume_path = config['resume_from'] or os.path.join(CKPT_DIR, 'last.pth')
if resume_path and os.path.exists(resume_path):
    # Note the scheduler is deliberately *not* restored from the checkpoint. timm's
    # CosineLRScheduler is a pure function of the epoch, so step(start_epoch) below fully
    # determines the LR - whereas loading its state_dict would also restore the old `t_initial`
    # and silently clobber the schedule if you raise config['epochs'] to extend training.
    ckpt = load_checkpoint(resume_path, model, optimizer, scheduler=None, scaler=scaler,
                           ema=model_ema, device=DEVICE)
    start_epoch = ckpt.get('epoch', -1) + 1
    best_top1 = max(ckpt.get('metrics', {}).get('best_top1', 0.0), 0.0)

    loaded_history = ckpt.get('history') or {}
    if set(loaded_history) >= set(history):
        history = {k: list(loaded_history[k]) for k in history}

    # Re-place the scheduler at the resumed epoch, as DeiT does
    # (`lr_scheduler.step(args.start_epoch)`). This also overwrites the stale LR that came
    # back with the optimizer state.
    scheduler.step(start_epoch)

    print(f"Resuming at epoch {start_epoch + 1}/{config['epochs']} "
          f"(best top-1 so far: {best_top1:.2f}%), lr {optimizer.param_groups[0]['lr']:.3e}")
else:
    print("No checkpoint found - training from scratch (random init).")

# Training

The loop stops when it hits `config['epochs']` **or** `config['max_runtime_hours']`, whichever
comes first, and always leaves a valid `last.pth` behind.

What to expect on Kaggle `T4 x2` at 224px with `batch_size=128, accum=2`:

- ~4-6 min/epoch (data loading on 4 vCPUs is often the bottleneck, not the GPU),
- ~60-90 epochs per 9-hour session,
- top-1 around 45-55% at epoch 30, 65-72% at epoch 100, ~78-80% at 300 epochs.

Tips if it is too slow: drop `image_size` to 160 (≈2x faster, then fine-tune at 224), or lower
`epochs` and accept a weaker model.

In [ ]:
if 'run_start_time' not in globals():
    run_start_time = time.time()

limit_batches = 20 if config['smoke_test'] else None
stop_reason = "completed"

for epoch in range(start_epoch, config['epochs']):
    print(f"\n=== Epoch {epoch + 1}/{config['epochs']} ===")
    epoch_start = time.time()
    curr_lr = optimizer.param_groups[0]['lr']

    if ra_sampler is not None:
        ra_sampler.set_epoch(epoch)      # reshuffles which images are repeated this epoch

    # ---------------- train ----------------
    train_top1, train_top5, train_loss = train_epoch(
        model, train_loader, optimizer, scaler, DEVICE, criterion,
        ema=model_ema,
        accum_steps=config['grad_accum_steps'],
        grad_clip=config['grad_clip'],
        epoch=epoch,
        limit_batches=limit_batches,
    )
    print(f"Train | top1 {train_top1:6.2f}% | top5 {train_top5:6.2f}% | "
          f"loss {train_loss:.4f} | lr {curr_lr:.3e}")

    # ---------------- validate ----------------
    val_top1, val_top5, val_loss = validate(
        model, val_loader, DEVICE, val_criterion, amp=config['amp'], desc=f"Val {epoch + 1}"
    )
    print(f"Val   | top1 {val_top1:6.2f}% | top5 {val_top5:6.2f}% | loss {val_loss:.4f}")

    ema_top1 = float('nan')
    if model_ema is not None:
        ema_top1, ema_top5, ema_loss = validate(
            model_ema.module, val_loader, DEVICE, val_criterion, amp=config['amp'], desc="Val EMA"
        )
        print(f"EMA   | top1 {ema_top1:6.2f}% | top5 {ema_top5:6.2f}% | loss {ema_loss:.4f}")

    # ---------------- bookkeeping ----------------
    score = max(val_top1, ema_top1 if model_ema is not None and not math.isnan(ema_top1) else 0.0)
    for key, value in [('epoch', epoch + 1), ('lr', curr_lr),
                       ('train_loss', train_loss), ('train_top1', train_top1),
                       ('val_loss', val_loss), ('val_top1', val_top1),
                       ('val_top5', val_top5), ('ema_top1', ema_top1)]:
        history[key].append(value)

    metrics = {'train_loss': train_loss, 'train_top1': train_top1, 'val_loss': val_loss,
               'val_top1': val_top1, 'val_top5': val_top5, 'ema_top1': ema_top1,
               'lr': curr_lr, 'best_top1': max(best_top1, score)}

    if run is not None:
        run.log(metrics, step=epoch + 1)

    with open(os.path.join(CKPT_DIR, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    last_path = os.path.join(CKPT_DIR, 'last.pth')
    save_checkpoint(last_path, model, optimizer, scheduler,
                    scaler, model_ema, epoch, metrics, history)
    if run is not None:
        wandb.save(last_path)

    if score > best_top1:
        best_top1 = score
        best_path = os.path.join(CKPT_DIR, 'best.pth')
        save_checkpoint(best_path, model, optimizer, scheduler,
                        scaler, model_ema, epoch, metrics, history)
        if run is not None:
            wandb.save(best_path)
        print(f"  -> new best top-1: {best_top1:.2f}% (saved best.pth)")

    # Step the LR schedule for the next epoch, after the checkpoint is safely written.
    scheduler.step(epoch + 1)

    elapsed = time.time() - run_start_time
    print(f"  epoch time {format_time(time.time() - epoch_start)} | "
          f"session elapsed {format_time(elapsed)} | best {best_top1:.2f}%")

    # ---------------- session time guard ----------------
    if elapsed > config['max_runtime_hours'] * 3600:
        stop_reason = "time budget reached"
        print(f"\nStopping early: {stop_reason}. Resume from "
              f"{os.path.join(CKPT_DIR, 'last.pth')} in the next session.")
        break

print(f"\nTraining finished ({stop_reason}). Best val top-1: {best_top1:.2f}%")
if run is not None:
    run.finish()

# Results

In [ ]:
hist = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))

axes[0].plot(hist['epoch'], hist['train_loss'], label='train')
axes[0].plot(hist['epoch'], hist['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(hist['epoch'], hist['train_top1'], label='train top-1')
axes[1].plot(hist['epoch'], hist['val_top1'], label='val top-1')
if hist['ema_top1'].notna().any():
    axes[1].plot(hist['epoch'], hist['ema_top1'], label='EMA top-1', ls='--')
axes[1].plot(hist['epoch'], hist['val_top5'], label='val top-5', alpha=.6)
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(hist['epoch'], hist['lr'])
axes[2].set_title('Learning rate'); axes[2].set_xlabel('epoch'); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.show()
hist.tail(10)

Note the **train top-1 below val top-1** pattern that is normal here: training accuracy is
measured on MixUp/CutMix-corrupted images against the original labels, so it understates the
model. A *large* and growing gap in the other direction is the signal to worry about.

# Final Evaluation

Loads `best.pth`, then reports:

1. plain center-crop top-1 / top-5,
2. horizontal-flip TTA (usually +0.2-0.5 top-1 for free),
3. the best and worst classes,
4. a grid of predictions on validation images.

In [ ]:
best_path = os.path.join(CKPT_DIR, 'best.pth')
if os.path.exists(best_path):
    ckpt = load_checkpoint(best_path, model, device=DEVICE)
    # Prefer the EMA weights if they were ahead at the best epoch
    m = ckpt.get('metrics', {})
    if ckpt.get('ema_state_dict') is not None and \
       (m.get('ema_top1') or 0) > (m.get('val_top1') or 0):
        unwrap(model).load_state_dict(ckpt['ema_state_dict'])
        print("Using EMA weights for evaluation.")
else:
    print("best.pth not found - evaluating the in-memory model.")

val_top1, val_top5, val_loss, preds, targets = validate(
    model, val_loader, DEVICE, val_criterion, amp=config['amp'],
    desc="Final val", return_preds=True,
)
print(f"\nFinal | top-1 {val_top1:.2f}% | top-5 {val_top5:.2f}% | loss {val_loss:.4f}")

In [ ]:
@torch.no_grad()
def validate_tta(model, dataloader, device, amp=True):
    """Average logits over the image and its horizontal flip."""
    model.eval()
    top1, top5 = AverageMeter(), AverageMeter()
    all_probs, all_targets = [], []

    for images, labels in tqdm(dataloader, desc="TTA", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=amp and device == 'cuda'):
            logits = forward_logits(model, images).float().softmax(1)
            logits = logits + forward_logits(model, torch.flip(images, dims=[3])).float().softmax(1)
        logits = logits / 2

        a1, a5 = topk_accuracy(logits, labels, topk=(1, 5))
        top1.update(a1.item(), images.size(0))
        top5.update(a5.item(), images.size(0))
        all_probs.append(logits.cpu())
        all_targets.append(labels.cpu())

    return top1.avg, top5.avg, torch.cat(all_probs), torch.cat(all_targets)


tta_top1, tta_top5, tta_probs, tta_targets = validate_tta(model, val_loader, DEVICE, config['amp'])
print(f"hflip TTA | top-1 {tta_top1:.2f}% | top-5 {tta_top5:.2f}%  "
      f"(vs {val_top1:.2f}% / {val_top5:.2f}% without TTA)")

In [ ]:
# Per-class accuracy
per_class = pd.DataFrame({'target': targets, 'pred': preds})
per_class['correct'] = (per_class.target == per_class.pred).astype(float)
acc_by_class = per_class.groupby('target')['correct'].mean().mul(100)
acc_by_class.index = [str(idx_to_name[i])[:28] for i in acc_by_class.index]

print("Easiest classes:\n", acc_by_class.sort_values(ascending=False).head(10).round(1), "\n")
print("Hardest classes:\n", acc_by_class.sort_values().head(10).round(1))

fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(range(len(acc_by_class)), acc_by_class.sort_values().values)
ax.axhline(val_top1, color='r', ls='--', label=f'mean {val_top1:.1f}%')
ax.set_xlabel('class (sorted)'); ax.set_ylabel('top-1 (%)')
ax.set_title('Per-class validation accuracy'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Qualitative check: predictions on a validation batch
images, labels = next(iter(val_loader))
with torch.no_grad(), torch.amp.autocast('cuda', enabled=config['amp'] and DEVICE == 'cuda'):
    logits = forward_logits(model, images[:8].to(DEVICE)).float()
pred = logits.argmax(1).cpu()

fig, axes = plt.subplots(1, 8, figsize=(18, 2.9))
vis = denormalize(images[:8])
for i, ax in enumerate(axes):
    ax.imshow(vis[i].permute(1, 2, 0).numpy())
    ok = pred[i] == labels[i]
    ax.set_title(f"{str(idx_to_name[int(pred[i])])[:14]}\n({str(idx_to_name[int(labels[i])])[:14]})",
                 fontsize=7, color='green' if ok else 'red')
    ax.axis('off')
plt.suptitle("prediction (ground truth)", fontsize=9)
plt.tight_layout(); plt.show()

# Export Artifacts

Writes everything you need for a report or the next session into `ENV['work_dir']`
(`/kaggle/working`, your Drive folder, or `./work`):

- `predictions_val.csv` - per-image prediction, target and top-5,
- `history.json` / `history.csv` - full training curves,
- `deit_s_imagenet100.pth` - weights plus the config and class list,
- `README.md` - a short model card.

On **Kaggle**, click **Save Version** afterwards so the output is persisted. On **Colab** these
files are already on Drive and need no further action.

In [ ]:
OUT_DIR = ENV['work_dir']

# ---- per-image predictions ----
top5_idx = tta_probs.topk(5, dim=1).indices.numpy()
pred_df = pd.DataFrame({
    'path'  : [p for p, _ in val_dataset.samples],
    'target': targets,
    'target_name': [str(idx_to_name[int(t)]) for t in targets],
    'pred'  : preds,
    'pred_name': [str(idx_to_name[int(p)]) for p in preds],
    'correct': (targets == preds).astype(int),
    'top5'  : [' '.join(map(str, row)) for row in top5_idx],
})
pred_df.to_csv(f"{OUT_DIR}/predictions_val.csv", index=False)

# ---- curves ----
hist.to_csv(f"{OUT_DIR}/history.csv", index=False)
with open(f"{OUT_DIR}/history.json", 'w') as f:
    json.dump(history, f, indent=2)

# ---- portable weights ----
torch.save({
    'model_state_dict': unwrap(model).state_dict(),
    'config': config,
    'classes': classes,
    'val_top1': val_top1,
    'val_top5': val_top5,
    'tta_top1': tta_top1,
}, f"{OUT_DIR}/deit_s_imagenet100.pth")

print("predictions_val.csv rows:", len(pred_df))
print(pred_df.head())

In [ ]:
MODEL = unwrap(model)   # the exact model behind the reported numbers

n_params = sum(p.numel() for p in MODEL.parameters())
README = f"""
# DeiT-S from scratch on ImageNet-100

- **Model**: DeiT-S / ViT-S-16 implemented from scratch with basic PyTorch modules
  (patch 16, dim {config['embed_dim']}, depth {config['depth']}, {config['num_heads']} heads,
  MLP ratio {config['mlp_ratio']}, {n_params / 1e6:.2f}M params). No pre-trained weights,
  no distillation teacher. Fused scaled-dot-product attention, pre-norm blocks,
  truncated-normal init, stochastic depth {config['drop_path_rate']}.
- **Data**: ImageNet-100 ({len(train_dataset):,} train / {len(val_dataset):,} val, 100 classes)
  at {config['image_size']}px.
- **Training** (DeiT's recipe via timm): AdamW (lr {config['lr']:.2e} =
  {config['base_lr']} x {config['eff_batch']}/512, wd {config['weight_decay']}, no decay on
  norms/biases/tokens/pos-embed), timm `CosineLRScheduler` stepped per epoch with a
  {config['warmup_epochs']}-epoch warm-up from {config['warmup_lr']} and cosine decay to
  {config['min_lr']}, {config['epochs']} epochs, effective batch {config['eff_batch']}
  ({config['batch_size']} x {config['grad_accum_steps']} accumulation),
  {'fp16 AMP' if config['amp'] else 'fp32'}, grad clip {config['grad_clip']},
  weight EMA (decay {config['ema_decay']}).
- **Loss**: `SoftTargetCrossEntropy` on Mixup's smoothed soft targets for training,
  plain cross-entropy for validation.
- **Augmentations** (timm `create_transform`, DeiT arguments): RandomResizedCrop
  ({config['image_size']}, scale {config['rrc_scale']}, bicubic), horizontal flip,
  auto-augment `{config['auto_augment']}`,
  RandomErasing(p={config['re_prob']}, mode={config['re_mode']}),
  MixUp({config['mixup_alpha']}) / CutMix({config['cutmix_alpha']}) switched 50/50 with
  label smoothing {config['label_smoothing']},
  repeated augmentation {config['ra_repeats']}x {'on' if config['repeated_aug'] else 'off'}.
- **Results**: val top-1 {val_top1:.2f}%, top-5 {val_top5:.2f}%, hflip-TTA top-1 {tta_top1:.2f}%.
- **Notebook execution**: runs on Kaggle, Colab, PSC Bridges-2 or locally - set the `PLATFORM`
  flag in the first cell (`'auto'` detects it) and the dataset, checkpoint and output paths are
  wired up automatically. Ran on {ENV['platform']} with work_dir `{ENV['work_dir']}`. The run
  checkpoints every epoch and stops at `max_runtime_hours`; re-run to resume (on Kaggle, point
  `config['resume_from']` at the saved `last.pth`).
"""

with open(f"{OUT_DIR}/README.md", 'w') as f:
    f.write(README)
print(README)

# Ideas for Pushing Accuracy Further

1. **More epochs.** This is the single biggest lever for ViTs from scratch. 300 epochs is the
   DeiT reference; 100 leaves several points on the table. Resume across sessions.
2. **Progressive resizing.** Train at 160px for most of the schedule, then fine-tune the last
   ~20 epochs at 224px with a low LR. Roughly halves total GPU time for similar accuracy
   (you need to interpolate `pos_embed` when the resolution changes).
3. **Knowledge distillation (the actual DeiT contribution).** Set `config['distilled'] = True`,
   train a ResNet/ConvNeXt teacher on ImageNet-100 first, and add the hard-distillation loss
   `0.5 * CE(student_cls, y) + 0.5 * CE(student_dist, teacher_argmax)`. The paper gets ~+1-2
   top-1 this way, and the gain is larger in low-data regimes like this one.
4. **Tune what the 1k recipe cannot carry over.** Three hyperparameters are tied to dataset
   scale rather than to the architecture: the number of epochs, the EMA horizon (see the config
   note) and repeated augmentation. Try `repeated_aug=False` at a fixed wall-clock budget - with
   130k images the 3x repeat may cost more in distinct-image throughput than it buys in
   regularization, which is the opposite of the 1.28M-image regime DeiT tuned it on.
5. **LayerScale + deeper models (CaiT) or a 4-stage hierarchy (Swin)** if the parameter budget
   allows - both close part of the data-efficiency gap with CNNs.
6. **Self-supervised pre-training on the same 130k images** (DINO / MAE) followed by supervised
   fine-tuning: no external data, but much better initialization than random.